# Partie commune — Architecture MLOps

**Objectif** : proposer une architecture MLOps complète pour industrialiser les deux modèles (détection de fraude + segmentation client).

**Plan :**
1. Pipeline de données (ingestion → validation → nettoyage)
2. Versionning (données, modèles, paramètres)
3. Architecture de déploiement (FastAPI + Docker)
4. Monitoring & détection de dérive
5. CI/CD avec GitHub Actions

## 1. Pipeline de données

```
┌─────────────────────────────────────────────────────────────┐
│                    PIPELINE DE DONNÉES                       │
│                                                              │
│  [Source]  →  [Ingestion]  →  [Validation]  →  [Nettoyage]  │
│  CSV/DB        pandas          Great            preprocessing│
│  Stream        DVC             Expectations     src/          │
│                                                              │
│                    ↓                                         │
│           [Feature Engineering]  →  [Split]  →  [Training]  │
│           src/preprocessing.py    sklearn       MLflow        │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
import sys
sys.path.append('../')

import pandas as pd
import numpy as np
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Pipeline de validation avec Great Expectations
# Exemple de règles de validation pour detection_fraude.csv

FRAUD_SCHEMA = {
    "required_columns": ["step", "type", "amount", "nameOrig", "oldbalanceOrg",
                          "newbalanceOrig", "nameDest", "oldbalanceDest",
                          "newbalanceDest", "isFraud"],
    "type_checks": {
        "amount": "float",
        "isFraud": "int",
    },
    "value_checks": {
        "amount": {"min": 0},
        "isFraud": {"allowed": [0, 1]},
        "type": {"allowed": ["PAYMENT", "TRANSFER", "CASH_OUT", "DEBIT", "CASH_IN"]},
    }
}


def validate_dataset(df: pd.DataFrame, schema: dict) -> dict:
    """Validation légère du dataset avant entraînement."""
    report = {"passed": True, "errors": []}

    # Colonnes manquantes
    missing = set(schema["required_columns"]) - set(df.columns)
    if missing:
        report["errors"].append(f"Colonnes manquantes : {missing}")
        report["passed"] = False

    # Vérifications de valeurs
    for col, checks in schema.get("value_checks", {}).items():
        if col not in df.columns:
            continue
        if "min" in checks and (df[col] < checks["min"]).any():
            n = (df[col] < checks["min"]).sum()
            report["errors"].append(f"{col} : {n} valeurs < {checks['min']}")
        if "allowed" in checks:
            invalid = ~df[col].isin(checks["allowed"])
            if invalid.any():
                report["errors"].append(f"{col} : valeurs invalides {df[col][invalid].unique()[:5]}")

    # Valeurs manquantes
    nulls = df[schema["required_columns"]].isnull().sum()
    if nulls.any():
        report["warnings"] = f"Valeurs manquantes : {nulls[nulls > 0].to_dict()}"

    return report


# Test sur un sample
sample = pd.DataFrame({
    "step": [1], "type": ["PAYMENT"], "amount": [100.0],
    "nameOrig": ["C123"], "oldbalanceOrg": [500.0], "newbalanceOrig": [400.0],
    "nameDest": ["M456"], "oldbalanceDest": [0.0], "newbalanceDest": [100.0],
    "isFraud": [0]
})

result = validate_dataset(sample, FRAUD_SCHEMA)
print("Validation :" , "✓ OK" if result["passed"] else "✗ ERREURS")
if result["errors"]:
    for e in result["errors"]:
        print(f"  - {e}")

## 2. Versionning avec MLflow

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score

# Configuration MLflow (local)
mlflow.set_tracking_uri("../mlruns")
mlflow.set_experiment("fraud_detection")

# Simulation d'un run MLflow
X_demo, y_demo = make_classification(n_samples=1000, n_features=10,
                                      weights=[0.99, 0.01], random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_demo, y_demo, test_size=0.2, random_state=42)

with mlflow.start_run(run_name="random_forest_v1"):
    params = {"n_estimators": 100, "max_depth": 10, "class_weight": "balanced"}
    mlflow.log_params(params)

    model = RandomForestClassifier(**params, random_state=42)
    model.fit(X_tr, y_tr)

    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]

    metrics = {
        "roc_auc": round(roc_auc_score(y_te, y_proba), 4),
        "f1": round(f1_score(y_te, y_pred), 4),
    }
    mlflow.log_metrics(metrics)
    mlflow.sklearn.log_model(model, "random_forest")

    print("Run MLflow enregistré :")
    for k, v in metrics.items():
        print(f"  {k}: {v}")

## 3. Architecture de déploiement

```
┌───────────────────────────────────────────────────────┐
│                  ARCHITECTURE DÉPLOIEMENT              │
│                                                        │
│   Client  →  [FastAPI]  →  [Model Registry]           │
│   (JSON)     port 8000     MLflow / joblib             │
│                                                        │
│   [Docker]  =  FastAPI + modèles + requirements        │
│   [Docker Compose]  =  API + MLflow UI + Monitoring    │
│                                                        │
│   [Streamlit Dashboard]  port 8501                     │
│     - Visualisation des prédictions                    │
│     - Suivi des métriques en temps réel                │
│     - Profils clients (clustering)                     │
└───────────────────────────────────────────────────────┘
```

Voir : [`mlops/api/app.py`](../mlops/api/app.py) et [`mlops/Dockerfile`](../mlops/Dockerfile)

## 4. Monitoring & Détection de dérive

In [ ]:
import matplotlib.pyplot as plt
from scipy import stats


def detect_data_drift(ref_data: pd.Series, new_data: pd.Series,
                       threshold_psi: float = 0.2) -> dict:
    """Population Stability Index (PSI) pour détecter la dérive."""
    bins = np.percentile(ref_data, np.linspace(0, 100, 11))
    bins[0] -= 1e-9
    bins[-1] += 1e-9

    ref_pct = np.histogram(ref_data, bins=bins)[0] / len(ref_data)
    new_pct = np.histogram(new_data, bins=bins)[0] / len(new_data)

    ref_pct = np.where(ref_pct == 0, 1e-6, ref_pct)
    new_pct = np.where(new_pct == 0, 1e-6, new_pct)

    psi = np.sum((new_pct - ref_pct) * np.log(new_pct / ref_pct))

    # KS-test
    ks_stat, ks_pvalue = stats.ks_2samp(ref_data, new_data)

    status = "stable" if psi < 0.1 else ("warning" if psi < threshold_psi else "drift")

    return {
        "psi": round(psi, 4),
        "ks_stat": round(ks_stat, 4),
        "ks_pvalue": round(ks_pvalue, 4),
        "status": status,
    }


# Simulation de dérive
np.random.seed(42)
ref = pd.Series(np.random.normal(1000, 500, 5000))   # données de référence
new_stable = pd.Series(np.random.normal(1050, 520, 1000))   # légère variation
new_drift = pd.Series(np.random.normal(2000, 800, 1000))    # dérive significative

print("=== Détection de dérive (feature 'amount') ===")
print("\nDonnées stables :")
print(detect_data_drift(ref, new_stable))
print("\nDonnées avec dérive :")
print(detect_data_drift(ref, new_drift))

In [ ]:
# Visualisation du monitoring
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Distribution comparée
axes[0].hist(ref, bins=40, alpha=0.6, color='steelblue', label='Référence', density=True)
axes[0].hist(new_drift, bins=40, alpha=0.6, color='tomato', label='Nouvelles données', density=True)
axes[0].set_title('Distribution : référence vs nouvelles données')
axes[0].legend()

# Simulation de métriques dans le temps
weeks = range(1, 25)
np.random.seed(0)
perf_stable = 0.95 + np.random.normal(0, 0.005, 24)
perf_drift = np.concatenate([0.95 + np.random.normal(0, 0.005, 12),
                               np.linspace(0.95, 0.80, 12) + np.random.normal(0, 0.01, 12)])

axes[1].plot(weeks, perf_stable, 'o-', color='steelblue', label='Stable', linewidth=1.5)
axes[1].plot(weeks, perf_drift, 'o-', color='tomato', label='Avec dérive', linewidth=1.5)
axes[1].axhline(0.90, color='orange', linestyle='--', label='Seuil alerte')
axes[1].axhline(0.85, color='red', linestyle='--', label='Seuil critique')
axes[1].set_title('Évolution ROC-AUC dans le temps')
axes[1].set_xlabel('Semaine')
axes[1].legend(fontsize=8)

# PSI dans le temps
psi_values = np.concatenate([np.random.uniform(0.02, 0.08, 12),
                               np.linspace(0.08, 0.35, 12)])
axes[2].bar(weeks, psi_values,
            color=['green' if p < 0.1 else 'orange' if p < 0.2 else 'red' for p in psi_values])
axes[2].axhline(0.1, color='orange', linestyle='--', label='Warning (0.1)')
axes[2].axhline(0.2, color='red', linestyle='--', label='Drift (0.2)')
axes[2].set_title('PSI (Population Stability Index)')
axes[2].set_xlabel('Semaine')
axes[2].legend(fontsize=8)

plt.suptitle('Dashboard de monitoring MLOps', y=1.02, fontsize=14)
plt.tight_layout()
plt.savefig('../reports/figures/mlops_monitoring.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. CI/CD avec GitHub Actions

Architecture du pipeline CI/CD :

```yaml
# .github/workflows/ml_pipeline.yml
#
# Déclencheur : push sur main ou pull_request
#
# Jobs :
#   1. test         → pytest tests/ (unit + integration)
#   2. lint         → flake8, black --check
#   3. train        → python scripts/train.py (si données disponibles)
#   4. validate     → comparaison métriques vs baseline MLflow
#   5. docker-build → docker build + push sur registry
#   6. deploy       → déploiement si métriques OK
```

Voir : [`.github/workflows/ml_pipeline.yml`](../.github/workflows/ml_pipeline.yml)

## Synthèse MLOps

| Composant | Outil | Rôle |
|-----------|-------|------|
| Versionning données | DVC | Suivi des versions CSV, reproductibilité |
| Versionning modèles | MLflow | Registre d'expériences, comparaison |
| Validation données | Great Expectations | Contrats de données, alertes qualité |
| API | FastAPI | Endpoints `/predict/fraud` et `/predict/segment` |
| Dashboard | Streamlit | Visualisation interactive pour les équipes métier |
| Containerisation | Docker | Reproductibilité, déploiement portable |
| Monitoring dérive | PSI + KS-test | Détection automatique de dérive |
| CI/CD | GitHub Actions | Tests, validation, déploiement automatique |

### Politique de réentraînement
- **Fraude** : déclenché si ROC-AUC < 0.90 **ou** PSI > 0.2 sur `amount` ou `type`
- **Clustering** : recalcul mensuel avec Silhouette Score de suivi
- **Validation automatique** : nouveau modèle déployé seulement si métriques > baseline − 2%